# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *63* |
| **Integrantes** | *Jorge Andrés Ocampo Suárez* |
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 19 de septiembre |
| **🎥 Enlace al video** | https://youtu.be/0-F_DGTOjwg |

**Nota:** Profesora, tuve un inconveniente con mi registro en la organización en GitHub, de igual manera publique mi trabajo en un repositorio.
https://github.com/jorgeocampoiudigital/bigdata-2026b-g063

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*Qué necesidad de infraestructura plantea el caso y qué debe soportar el entorno.*

---
## 2. Descripción de los datos

*Qué va a vivir en esta infraestructura: volumen esperado, frecuencia de actualización
y quién la va a consumir.*

---
## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura

*Fuentes → ingesta → almacenamiento → procesamiento → consumo.
Señalar explícitamente qué capa administra el proveedor y cuál el equipo.
Insertar la imagen o usar un diagrama en texto.*

```
[ fuentes ] → [ ingesta ] → [ almacenamiento ] → [ procesamiento ] → [ consumo ]
```
![](/Workspace/Users/jorge.ocampo@est.iudigital.edu.co/bigdata-2026b-g063/ea2/mermaid-diagram-1789859031600.png)

### 3.2 Matriz de roles

*Qué puede hacer cada rol sobre cada capa en un entorno real.*

| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| Analista | Sin acceso — datos crudos, sin garantías de calidad, no aptos para análisis de negocio | Solo lectura — consulta y explora datos ya limpios y tipados para construir reportes o responder preguntas ad hoc | Solo lectura — consume tablas agregadas/curadas pensadas para consumo directo (dashboards, métricas) |
| Ingeniero de datos | Lectura y escritura — responsable de la ingesta cruda y de mantener la trazabilidad de la fuente | Lectura y escritura — construye y mantiene las transformaciones, reglas de limpieza y validaciones de calidad | Lectura y escritura — construye las agregaciones y tablas de negocio a partir de plata |
| Administrador | Control total (lectura, escritura, permisos) — además administra quién más tiene acceso a cada capa | Control total | Control total |

Esta matriz sigue el principio de menor privilegio en función de para qué usa cada rol la información: el analista nunca necesita ver datos crudos sin limpiar (bronce) porque trabajar sobre ellos directamente puede llevar a conclusiones erróneas (duplicados, tipos sin castear, nulos sin tratar), así que su acceso empieza en plata; solo llega a acceder a oro y plata en modo lectura porque su función es consumir, no transformar. El ingeniero de datos necesita escritura en las tres capas porque es quien construye el pipeline completo (bronce → plata → oro), pero normalmente no necesita administrar permisos de otros usuarios — esa es responsabilidad exclusiva del administrador, que es el único rol con control total sobre las tres capas, incluyendo la gestión de quién más tiene acceso.

### 3.3 Especificación del equivalente IaaS

*Se diseña, no se implementa. Máquinas y dimensionamiento, sistema operativo,
software a instalar, red, almacenamiento, y estimación del esfuerzo de puesta
en marcha y de operación.*

**Máquinas y dimensionamiento**

| Rol | Cantidad | Especificación sugerida | Justificación |
|---|---|---|---|
| Nodo driver (Spark master) | 1 | 4 vCPU / 16 GB RAM | Coordina el clúster, no procesa datos pesados, pero necesita memoria para el plan de ejecución y metadata |
| Nodos worker (Spark) | 2–4 | 8 vCPU / 32 GB RAM c/u | Ejecutan las tareas distribuidas (shuffles, joins), para el tamaño de Wanderbricks (cientos de miles de filas), 2-4 workers son suficientes |
| Nodo de almacenamiento / metastore | 1 | 2 vCPU / 8 GB RAM | Aloja el metastore de Hive/catálogo y coordina metadata de las tablas Delta |

**Sistema operativo:** Ubuntu Server 22.04 LTS — base común para desplegar Spark manualmente, con buen soporte de paquetes Java/Python.

**Software a instalar en cada máquina**

- Java (JDK 11 o 17) — Spark corre sobre la JVM
- Apache Spark (misma versión en todos los nodos, para evitar incompatibilidades)
- Python 3.10+ y PySpark
- Un gestor de clúster: Spark Standalone o YARN/Kubernetes.
- Delta Lake (librería, para mantener el formato de tablas que ya se usa)
- Hive Metastore (o un metastore externo tipo PostgreSQL) para persistir el catálogo
- Herramientas de monitoreo (Ganglia o Prometheus + Grafana) — en Databricks viene incluido, aquí hay que montarlo aparte

**Red**

- Red privada (VPC) con subredes separando el driver/workers del acceso público
- Reglas de firewall abriendo solo los puertos necesarios entre nodos (7077 para Spark Standalone, 8080/4040 para UIs) y bloqueando el resto

**Almacenamiento**

- Discos SSD locales en cada worker para shuffle temporal (Spark escribe datos intermedios en disco durante los shuffles)
- Almacenamiento de objetos para las tablas Delta persistentes, montado o accedido vía conector — reemplaza lo que Unity Catalog administra de forma transparente en Databricks

**Estimación de esfuerzo**

| Actividad | Esfuerzo estimado | Por qué |
|---|---|---|
| Puesta en marcha inicial | 3-5 días-persona | Aprovisionar VMs, instalar y configurar Java/Spark/Delta en cada nodo, configurar red y metastore, validar que el clúster arranca y ejecuta un job de prueba |
| Configuración de seguridad y permisos | 1-2 días-persona | En Databricks esto es `GRANT`/`SHOW GRANTS`; aquí hay que implementar autenticación propia como Kerberos y control de acceso a nivel de sistema de archivos |
| Operación mensual (parches, monitoreo, reinicio de nodos caídos) | 1-2 días-persona/mes | Sin gestión automática, cada actualización de Spark o parche de seguridad del SO es manual, y hay que vigilar la salud del clúster manualmente |

Esta especificación deja claro por qué Databricks (PaaS) resulta atractivo frente a esta alternativa: todo lo que aquí requiere días de trabajo humano (aprovisionar, instalar, asegurar, monitorear) en Databricks es responsabilidad del proveedor. La contrapartida es que sobre IaaS se tiene control total sobre cada componente — versión exacta de Spark, configuración fina del clúster, ubicación física de los datos — control que en un modelo gestionado se cede a cambio de velocidad de despliegue.

### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS | PaaS (Databricks) | SaaS |
|---|---|---|---|
| **Control** | Total — se elige SO, versión exacta de Spark, configuración de red y almacenamiento | Parcial — se controla el código, los datos y los permisos, pero no la infraestructura subyacente ni el runtime | Mínimo — solo se configuran parámetros dentro de lo que la aplicación permite; no hay acceso al motor de procesamiento |
| **Tiempo hasta el primer resultado** | Días (aprovisionar VMs, instalar Java/Spark/Delta, configurar red y metastore antes de correr una sola consulta) | Minutos a horas (clúster gestionado, catálogo ya disponible; solo hay que escribir el pipeline) | Minutos (conectar la fuente de datos y usar la interfaz ya construida) |
| **Esfuerzo operativo** | Alto y continuo — parches de SO, actualizaciones de Spark, monitoreo de nodos, todo manual | Bajo — Databricks gestiona el runtime, el autoescalado y la disponibilidad del clúster | Prácticamente nulo — el proveedor administra todo, incluida la lógica de negocio de la aplicación |
| **Costo** | Se paga por VM encendida sin importar el uso real; requiere dimensionar de antemano y suele haber sobreaprovisionamiento | Se paga por cómputo efectivamente usado (DBU), con autoescalado que ajusta el gasto a la carga real | Suscripción fija por usuario/funcionalidad, independiente del volumen de procesamiento real |
| **Escalabilidad** | Manual — hay que aprovisionar y configurar cada nodo nuevo | Automática — el clúster escala según la carga sin intervención | Depende del proveedor; normalmente limitada por los planes de suscripción, no ajustable por el usuario |
| **Gobierno** | Se construye desde cero (autenticación propia, control de acceso a nivel de sistema de archivos) | Nativo e integrado — Unity Catalog con `GRANT`, roles y linaje ya incorporados | Limitado a lo que la aplicación expone; normalmente sin control fino sobre permisos a nivel de dato |

**Conclusión:** para el caso de Wanderbricks, **PaaS (Databricks) es la opción que conviene**. IaaS ofrecería más control, pero ese control no se traduce en ningún beneficio real para este proyecto, el equipo no necesita ajustar configuraciones de bajo nivel del motor Spark. El esfuerzo operativo que exige IaaS (parches, monitoreo, gestión manual de permisos) no aporta valor de negocio: es trabajo que hay que hacer para mantener la infraestructura funcionando, no para responder mejor las preguntas analíticas del caso. SaaS, en el otro extremo, sería insuficiente porque el proyecto exige escribir transformaciones propias, controlar el esquema de las tablas y ejecutar consultas SQL/PySpark a medida — cosas que una aplicación SaaS cerrada no permite. PaaS es el punto medio correcto: cede la infraestructura al proveedor y conserva el control sobre los datos, el código y los permisos.

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# Catálogo del grupo
CATALOGO = "bigdata_grupo63"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"USE CATALOG {CATALOGO}")

# Un esquema por capa, en vez de un solo esquema con todas las tablas mezcladas
spark.sql("CREATE SCHEMA IF NOT EXISTS raw")
spark.sql("CREATE SCHEMA IF NOT EXISTS wanderbricks")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Volumen para artefactos no tabulares (diagramas, capturas, logs del Job)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.wanderbricks.artefactos")

# Verificación
display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

La organización sigue un criterio de **separación por capa de madurez del dato**, no por tipo de tabla ni por integrante del equipo:

- **`raw`**: reservado para datos de entrada sin ningún procesamiento, útil si en el futuro se ingieren fuentes adicionales a Wanderbricks (por ejemplo, otro dataset externo) sin mezclarlos con el pipeline ya construido.
- **`wanderbricks`**: contiene las tablas bronce y plata ya construidas en EA1 (`bronze_*`, `silver_*`). Se mantiene el mismo esquema de EA1 en vez de migrar las tablas, porque el catálogo y esquema ya existentes cumplen la función de organización por capa que se necesita.
- **`gold`**: separado de `wanderbricks` a propósito, porque las tablas oro tienen un consumidor distinto (analistas de negocio, dashboards) al de bronce/plata (ingenieros de datos). Esta separación es la que sostiene la matriz de roles: los permisos de lectura para analistas se otorgan sobre `gold`, no sobre todo el catálogo.
- **Volumen `artefactos`**: los notebooks y Jobs no solo producen tablas, también generan archivos (diagramas exportados, logs, capturas). Un volumen dedicado evita que estos archivos se mezclen con la lógica de datos tabulares y facilita referenciarlos desde el código con una ruta estable (`/Volumes/bigdata_grupo63/wanderbricks/artefactos/`).

Este criterio de separación por capa, es el que hace posible que los `GRANT` sean simples y explicables: cada rol recibe permisos sobre el esquema que le corresponde según su función, no sobre tablas sueltas dispersas.

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
# Grupos disponibles en la plataforma gratuita
GRUPO_TODOS = "account users"

# GRANT 1 — lectura sobre la capa oro para todo el grupo de usuarios (rol analista)
# Simula el acceso de solo lectura que tendría un analista en un entorno real
spark.sql(f"""
    GRANT SELECT ON SCHEMA {CATALOGO}.gold TO `{GRUPO_TODOS}`
""")

# GRANT 2 — lectura y escritura sobre bronce (rol ingeniero de datos)
# Se otorga al mismo grupo pero sobre un objeto y nivel distinto, para poder
# documentar y explicar la diferencia aunque la plataforma no separe roles reales
spark.sql(f"""
    GRANT SELECT, MODIFY ON SCHEMA {CATALOGO}.wanderbricks TO `{GRUPO_TODOS}`
""")

# Verificación de ambos permisos
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.gold"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.wanderbricks"))

La plataforma gratuita solo expone el grupo `account users`, así que los dos `GRANT` se aplican sobre ese mismo grupo pero con **nivel y objeto distintos**, para poder documentar y sustentar la diferencia aunque el entorno no permita separar roles reales:

1. **`GRANT SELECT` sobre el esquema `gold`**: simula el acceso de un analista, solo lectura sobre datos ya curados.
2. **`GRANT SELECT, MODIFY` sobre el esquema `wanderbricks`** (bronce + plata): simula el acceso de un ingeniero de datos, lectura y escritura sobre las capas que construye y mantiene.

En un entorno real con Unity Catalog completo (no la capa gratuita), estos mismos `GRANT` se aplicarían a grupos separados (`analistas`, `ingenieros_datos`) en vez de a `account users`, exactamente como se especificó en la matriz de roles. La limitación aquí es de la plataforma, no del diseño.

### 4.3 Linaje

*Evidenciar el linaje de una tabla desde el Catalog Explorer. Insertar la captura.*

![](/Workspace/Users/jorge.ocampo@est.iudigital.edu.co/bigdata-2026b-g063/ea2/linaje.png)

La captura muestra el linaje de `silver_bookings` desde el Catalog Explorer: se origina en `bronze_bookings` (ingesta cruda de `samples.wanderbricks`), y fue modificada posteriormente por el `MERGE` ejecutado en la EA1. Unity Catalog registra esta trazabilidad de forma automática, sin que el equipo tuviera que documentar manualmente qué notebook o proceso tocó cada tabla.

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

In [0]:
# Tarea 1: refrescar la capa bronce (re-ingesta desde samples.wanderbricks)
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"
tablas = [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]

for t in tablas:
    (spark.table(f"samples.wanderbricks.{t}")
        .withColumn("_ingested_at", F.current_timestamp())
        .write.format("delta").mode("overwrite")
        .saveAsTable(f"{CATALOGO}.wanderbricks.bronze_{t}"))

print("Bronce actualizado:", tablas)

In [0]:
# Tarea 2: reconstruir la capa plata a partir de bronce (depende de que la Tarea 1 termine)
CATALOGO = "bigdata_grupo63"

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOGO}.wanderbricks.silver_bookings AS
    SELECT booking_id, user_id, property_id,
           CAST(check_in AS DATE), CAST(check_out AS DATE),
           CAST(guests_count AS INT), CAST(total_amount AS DECIMAL(10,2)),
           status, created_at, updated_at
    FROM {CATALOGO}.wanderbricks.bronze_bookings
    WHERE total_amount >= 0 AND check_out > check_in
""")

print("Plata actualizada: silver_bookings")

**Job ID:** 523402855912221 

![](/Workspace/Users/jorge.ocampo@est.iudigital.edu.co/bigdata-2026b-g063/ea2/ejecucion_job.png)


El Job `wanderbricks_pipeline_diario` encadena dos tareas: `refrescar_bronce` (re-ingesta desde `samples.wanderbricks`) y `reconstruir_plata` (reconstruye `silver_bookings` aplicando las reglas de calidad definidas en EA1), con una dependencia explícita entre ambas, la segunda tarea solo se ejecuta si la primera termina exitosamente. Se programó con una periodicidad diaria, simulando el mantenimiento continuo que tendría este pipeline en producción. La captura muestra una ejecución exitosa de ambas tareas.

---
## 5. Resultados

El entorno quedó organizado en tres esquemas dentro del catálogo `bigdata_grupo63`, separados por función (`raw` para futuras fuentes, `wanderbricks` para bronce/plata, `gold` para consumo), lo que permitió que los permisos otorgados fueran directos de justificar: `SELECT` sobre `gold` y `SELECT, MODIFY` sobre `wanderbricks`, reflejando la matriz de roles definida dentro de las limitaciones del grupo único (`account users`) que ofrece la plataforma gratuita.

El linaje de `silver_bookings` quedó documentado de forma automática por Unity Catalog, mostrando su origen en `bronze_bookings` sin que el equipo tuviera que mantener esa trazabilidad manualmente.

La automatización se validó con una ejecución exitosa del Job `wanderbricks_pipeline_diario`, con sus dos tareas encadenadas (`refrescar_bronce` → `reconstruir_plata`) completadas correctamente y una programación diaria definida, cumpliendo el requisito de un pipeline reproducible sin intervención manual.

En conjunto, estos resultados muestran en la práctica lo que la comparación planteaba en teoría: organizar, asegurar, trazar y automatizar este entorno tomó minutos de configuración sobre Databricks PaaS, mientras que en IaaS estimó 3-5 días-persona solo para tener un clúster equivalente funcionando, antes de siquiera empezar a aplicar permisos o automatizaciones.

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

- El costo de gobernar y automatizar un entorno de datos cambia radicalmente según el modelo de despliegue. Organizar el catálogo en tres esquemas, aplicar dos niveles de permiso distintos, trazar el linaje de una tabla y dejar un pipeline programado corriendo tomó una sesión de trabajo sobre Databricks y cada una de esas cuatro tareas, especificadas para IaaS, habría requerido antes aprovisionar y asegurar máquinas virtuales desde cero. Esa diferencia de esfuerzo, no la disponibilidad de las funciones en sí, es el argumento central a favor de PaaS para este caso.

- Lo que no funcionó exactamente como se esperaba fue la separación de roles: la plataforma gratuita solo expone el grupo `account users`, así que los dos `GRANT` tuvieron que aplicarse sobre el mismo grupo con niveles distintos, en vez de sobre grupos separados como plantea la matriz de roles. Esto no invalida el diseño, la matriz sigue siendo el criterio correcto para un entorno real con licencias completas de Unity Catalog, pero es una limitación de la plataforma que hay que declarar explícitamente y no disimular como si los roles estuvieran realmente separados.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Jorge Andrés Ocampo Suárez | Todas las sesiones | Todas las sesiones |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
    El proveedor administra el cómputo y el almacenamiento físico. Nosotros administramos el diseño del catálogo, los permisos, y qué corre en el Job.
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
    Se muestra en el apartado de roles, GRANT da `SELECT, MODIFY` sobre el esquema `wanderbricks` al grupo disponible, simulando el rol de ingeniero de datos, necesita escribir porque es quien ejecuta la ingesta y las transformaciones.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?
    Lo primero sería el tiempo de puesta en marcha: instalar y configurar Java, Spark y Delta Lake de forma consistente en cada nodo, y montar un metastore propio, antes de poder ejecutar una sola consulta, algo que en Databricks ya viene resuelto desde el primer minuto.

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas